# Picotron native 52M: DDP ZeRO pretraining → SFT → DPO

This Kaggle GPU notebook exercises the native `PicotronDecoderModel`, rather than an arbitrary Hugging Face model. It combines GQA, mixed RoPE/NoPE, MoE, sliding-window attention, activation checkpointing, DDP, ZeRO stage 1, safetensors checkpoints, persistent CSV/log output, SFT, and DPO.

**Important:** this notebook was authored without GPU or Hub execution. Run every cell on a Kaggle notebook with two T4 GPUs enabled.

## 1. Clone Picotron and install the package

The editable install provides the `picotron` CLI used by the DDP launch below.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/kaggle/working/picotron')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/AndyMLAndAI/picotron.git', str(REPO)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO, check=True)
# Dataset/tokenizer access used by preprocessing, SFT, and DPO.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets', 'transformers'], check=True)


## 2. Verify the two T4s and Picotron's Turing policy

A T4 is compute capability 7.5, so Picotron must choose fp16—not bf16.

In [ ]:
import torch
from picotron.utils.hardware import (
    get_gpu_compute_capability,
    select_attention_backend,
    select_training_dtype,
)

assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, 'Enable two Kaggle T4 GPUs.'
for device_index in range(torch.cuda.device_count()):
    capability = get_gpu_compute_capability(device_index)
    print(f'GPU {device_index}: {torch.cuda.get_device_name(device_index)}; capability={capability}')
    assert capability == (7, 5), 'This demo is deliberately sized for T4/Turing.'
assert select_training_dtype(0) is torch.float16
print('dtype:', select_training_dtype(0), 'attention backend:', select_attention_backend(0))


## 3. Preprocess 1B deduplicated FineWeb-Edu tokens

This invokes the reusable multiprocess preprocessing tool; it streams `HuggingFaceTB/smollm-corpus`'s `fineweb-edu-dedup` subset, caches raw text locally for resumability, and writes a uint16 GPT-2 token cache. The resulting raw cache is about 1.86 GiB; raw-text caching needs additional disk space.

In [ ]:
TARGET_TOKENS = 1_000_000_000
TOKEN_PATH = REPO / 'data' / 'fineweb_edu_dedup_1b_gpt2.uint16'
TEXT_CACHE = REPO / 'data' / 'raw_text_cache'
TOKEN_PATH.parent.mkdir(parents=True, exist_ok=True)

subprocess.run([
    sys.executable, 'tools/preprocess_data.py',
    '--dataset-name', 'HuggingFaceTB/smollm-corpus',
    '--dataset-config', 'fineweb-edu-dedup',
    '--tokenizer-name', 'gpt2',
    '--target-tokens', str(TARGET_TOKENS),
    '--output-path', str(TOKEN_PATH),
    '--text-field', 'text',
    '--tokenize-workers', '4',
    '--text-cache-dir', str(TEXT_CACHE),
], cwd=REPO, check=True)
assert TOKEN_PATH.stat().st_size == TARGET_TOKENS * 2
print(f'Prepared {TOKEN_PATH}: {TOKEN_PATH.stat().st_size / 2**30:.2f} GiB')


## 4. Write the native ~52M pretraining configuration

This is **Qwen-inspired, not Qwen3.5 itself**: Picotron does not implement Qwen3.5's GatedDeltaNet hybrid layers. It does exercise compatible ideas: GQA (8 query heads / 2 KV heads), RoPE with NoPE in the middle four layers, MoE in every block, and a 256-token local attention window.

Parameter estimate with tied GPT-2 embeddings: `50,257×512 = 25.73M` embeddings; each block is approximately `655,360` GQA attention + `2×(3×512×832) + 1,024` MoE = `3.21M`; eight blocks give `25.71M`. Total is approximately **51.44M** plus norms—reported exactly in the next cell.

Global tokens/step are `2 GPUs × micro_batch_size 4 × sequence_length 512 = 4,096`. Two 1B-token epochs require `2,000,000,000 / 4,096 = 488,282` steps. Even at an optimistic 8 steps/s that is ~17 hours; at 2 steps/s it is ~68 hours. This is unlikely to fit a single Kaggle session, so measure throughput with a short run and split the run using resume checkpoints.

In [ ]:
%%writefile /kaggle/working/picotron/config_pretrain_52m.yaml
checkpoints:
  checkpoint_interval: 5000
  checkpoints_path: /kaggle/working/picotron/checkpoints/native_52m_pretrain.safetensors
  resume_checkpoint_path: null
  load_optimizer: true
  save_final_state: true
model:
  # Turing/T4 rule: fp16 only; bf16 is never valid here.
  dtype: float16
  model_config:
    # ~51.44M parameters: tied 50,257x512 embedding plus 8 x ~3.21M blocks.
    vocab_size: 50257
    hidden_size: 512
    intermediate_size: 832
    num_hidden_layers: 8
    num_attention_heads: 8
    num_key_value_heads: 2
    attention_type: gqa
    rope_theta: 1000000.0
    # Keep RoPE at the boundary layers; remove explicit rotation in the middle.
    nope_layers: [2, 3, 4, 5]
    # Local attention bounds eager-attention activation memory on T4.
    sliding_window_size: 256
    # Every block is MoE so this run tests routing/aux-loss composition end-to-end.
    moe_config:
      num_experts: 2
      top_k: 1
      aux_loss_coefficient: 0.01
    tie_word_embeddings: true
    gradient_checkpointing: true
optimizer:
  learning_rate_scheduler:
    learning_rate: 0.0003
  optimizer_factory:
    name: adamw
    adam_beta1: 0.9
    adam_beta2: 0.95
    adam_eps: 1.0e-08
  weight_decay: 0.1
  clip_grad: 1.0
parallelism:
  dp: 2
  # This is the project's canonical ZeRO setting; optimizer has no duplicate field.
  zero_stage: 1
tokens:
  sequence_length: 512
  micro_batch_size: 4
  # ceil(2 epochs x 1,000,000,000 tokens / (2 GPUs x 4 x 512)).
  train_steps: 488282
data:
  dataset_token_path: /kaggle/working/picotron/data/fineweb_edu_dedup_1b_gpt2.uint16
  tokenizer_name: gpt2
  vocab_size: 50257
  num_workers: 2
  prefetch_factor: 2
logging:
  log_level: INFO
  iteration_step_info_interval: 10
  file_logging: true
  file_logging_output_dir: /kaggle/working/picotron/logs
general:
  project: picotron
  run: native-52m-pretrain
  seed: 1337


In [ ]:
from picotron.config.config import load_config
from picotron.models.picotron_decoder import PicotronDecoderModel

config = load_config(REPO / 'config_pretrain_52m.yaml')
model_for_count = PicotronDecoderModel(config)
parameter_count = sum(parameter.numel() for parameter in model_for_count.parameters())
print(f'Native parameter count: {parameter_count:,} ({parameter_count / 1e6:.2f}M)')
assert 50_000_000 <= parameter_count <= 53_000_000
del model_for_count


## 5. Launch two-rank DDP + native ZeRO stage 1 pretraining

`torchrun` invokes the installed `picotron` entrypoint. Rank 0 writes the shared safetensors weights and metadata; each rank writes its own ZeRO optimizer-state shard. File logging is always enabled, so inspect `logs/native-52m-pretrain/metrics.csv` and `run.log` after interruption or completion.

In [ ]:
import os

# Set a deliberately smaller value first for a throughput/memory smoke test, then restore 488282.
# The YAML remains the reproducible full two-epoch plan.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
subprocess.run([
    'torchrun', '--standalone', '--nproc_per_node=2',
    subprocess.check_output(['bash', '-lc', 'command -v picotron'], text=True).strip(),
    '--config', 'config_pretrain_52m.yaml',
], cwd=REPO, check=True)

PRETRAIN_CHECKPOINT = REPO / 'checkpoints' / 'native_52m_pretrain.safetensors'
assert PRETRAIN_CHECKPOINT.exists()
assert (REPO / 'checkpoints' / 'native_52m_pretrain.optimizer.rank0.pt').exists()
assert (REPO / 'checkpoints' / 'native_52m_pretrain.optimizer.rank1.pt').exists()
print('Pretraining checkpoint:', PRETRAIN_CHECKPOINT)


## 6. Native greedy inference after pretraining

The native model has no KV-cache/generation API yet, so this simple greedy loop recomputes the prefix at each token. It is intended for short demo prompts only.

In [ ]:
from safetensors.torch import load_file
from transformers import AutoTokenizer

DEVICE = torch.device('cuda:0')
tokenizer = AutoTokenizer.from_pretrained('gpt2', use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_native(weights_path):
    active_model = PicotronDecoderModel(config).to(DEVICE)
    active_model.load_state_dict(load_file(str(weights_path), device='cuda:0'))
    active_model.eval()
    return active_model

def generate_native(active_model, prompt, max_new_tokens=64):
    token_ids = tokenizer.encode(prompt, add_special_tokens=False)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            context = torch.tensor([token_ids[-config.tokens.sequence_length:]], device=DEVICE)
            next_token = active_model(context)[0, -1].argmax().item()
            token_ids.append(next_token)
            if next_token == tokenizer.eos_token_id:
                break
    return tokenizer.decode(token_ids)

PROMPTS = [
    'Explain why leaves change color in autumn.',
    'Write a Python function that returns the nth Fibonacci number.',
]
pretrained_model = load_native(PRETRAIN_CHECKPOINT)
pretrain_outputs = [generate_native(pretrained_model, prompt) for prompt in PROMPTS]
for prompt, output in zip(PROMPTS, pretrain_outputs):
    print(f'PROMPT: {prompt}\nPRETRAINED: {output}\n')


## 7. Write SFT settings and prepare a native-model SmolTalk dataset

The YAML is the reproducibility record. The notebook uses `SFTTrainer` directly because native Picotron needs a `(input_ids, labels)` dataset and the SFT CLI's factory hooks are intentionally general-purpose. This demo uses 10,000 streamed examples and 2,000 steps; reduce those counts for a first GPU smoke test.

In [ ]:
%%writefile /kaggle/working/picotron/config_sft_52m.yaml
base_config_path: config_pretrain_52m.yaml
base_checkpoint_path: checkpoints/native_52m_pretrain.safetensors
dataset_path: artifacts/smoltalk_10k.jsonl
max_steps: 2000


In [ ]:
import itertools
from datasets import load_dataset
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from picotron.serialize.checkpoint import save_checkpoint
from picotron_sft import SFTTrainer

SFT_EXAMPLES = 10_000
SFT_STEPS = 2_000
smoltalk_stream = load_dataset('HuggingFaceTB/smoltalk', 'all', split='train', streaming=True)
smoltalk_examples = list(itertools.islice(smoltalk_stream, SFT_EXAMPLES))
assert len(smoltalk_examples) == SFT_EXAMPLES

class NativeSmolTalkDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        messages = self.examples[index]['messages']
        text = '\n'.join(f"{message['role']}: {message['content']}" for message in messages)
        ids = tokenizer.encode(text, add_special_tokens=False)[: config.tokens.sequence_length - 1]
        ids = ids + [tokenizer.eos_token_id]
        padding = config.tokens.sequence_length - len(ids)
        input_ids = ids + [tokenizer.pad_token_id] * padding
        labels = ids + [-100] * padding
        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

sft_loader = DataLoader(NativeSmolTalkDataset(smoltalk_examples), batch_size=2, shuffle=True)
sft_model = load_native(PRETRAIN_CHECKPOINT).train()
sft_optimizer = AdamW(sft_model.parameters(), lr=2e-5, weight_decay=0.01)
sft_trainer = SFTTrainer(
    sft_model,
    sft_loader,
    learning_rate=2e-5,
    num_steps=SFT_STEPS,
    optimizer=sft_optimizer,
    device=DEVICE,
    display_config=config,
)
sft_losses = sft_trainer.train()
SFT_CHECKPOINT = REPO / 'checkpoints' / 'native_52m_smoltalk.safetensors'
save_checkpoint(sft_model, sft_optimizer, len(sft_losses), SFT_CHECKPOINT)
print(f'SFT: first_loss={sft_losses[0]:.4f}, last_loss={sft_losses[-1]:.4f}')


## 8. Inference after SFT

Use the exact same prompts so qualitative changes remain attributable to the stage transition.

In [ ]:
sft_model.eval()
sft_outputs = [generate_native(sft_model, prompt) for prompt in PROMPTS]
for prompt, output in zip(PROMPTS, sft_outputs):
    print(f'PROMPT: {prompt}\nSFT: {output}\n')


## 9. Write DPO settings and prepare UltraFeedback preferences

DPO creates a frozen copy of the SFT policy as its reference. This bounded demo streams 5,000 preference triples and runs 1,000 updates; it should be reduced for a first memory smoke test.

In [ ]:
%%writefile /kaggle/working/picotron/config_dpo_52m.yaml
base_config_path: config_pretrain_52m.yaml
base_checkpoint_path: checkpoints/native_52m_smoltalk.safetensors
dataset_path: artifacts/ultrafeedback_5k.jsonl
beta: 0.1
max_steps: 1000


In [ ]:
from picotron_dpo import DPOTrainer
from picotron_dpo.data import PreferenceDataset, collate_preference_batch, infer_pad_token_id

DPO_EXAMPLES = 5_000
DPO_STEPS = 1_000

def last_assistant_content(messages):
    for message in reversed(messages):
        if message.get('role') == 'assistant' and isinstance(message.get('content'), str):
            return message['content']
    return None

def format_dpo_prompt(prompt):
    return f'user: {prompt}\nassistant: '

preference_stream = load_dataset(
    'HuggingFaceH4/ultrafeedback_binarized', split='train_prefs', streaming=True
)
dpo_triples = []
for example in preference_stream:
    chosen = last_assistant_content(example['chosen'])
    rejected = last_assistant_content(example['rejected'])
    if isinstance(example['prompt'], str) and chosen and rejected:
        dpo_triples.append((format_dpo_prompt(example['prompt']), chosen, rejected))
    if len(dpo_triples) == DPO_EXAMPLES:
        break
assert len(dpo_triples) == DPO_EXAMPLES

preference_dataset = PreferenceDataset(
    dpo_triples, tokenizer, max_length=config.tokens.sequence_length
)
dpo_loader = DataLoader(
    preference_dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=lambda examples: collate_preference_batch(
        examples, pad_token_id=infer_pad_token_id(tokenizer)
    ),
)
# Re-load SFT weights so DPO's automatically copied reference is exactly the SFT policy.
dpo_model = load_native(SFT_CHECKPOINT).train()
dpo_optimizer = AdamW(dpo_model.parameters(), lr=1e-5, weight_decay=0.01)
dpo_trainer = DPOTrainer(
    dpo_model,
    dpo_loader,
    beta=0.1,
    learning_rate=1e-5,
    num_steps=DPO_STEPS,
    optimizer=dpo_optimizer,
    device=DEVICE,
    display_config=config,
)
dpo_losses = dpo_trainer.train()
DPO_CHECKPOINT = REPO / 'checkpoints' / 'native_52m_ultrafeedback_dpo.safetensors'
save_checkpoint(dpo_model, dpo_optimizer, len(dpo_losses), DPO_CHECKPOINT)
print(f'DPO: first_loss={dpo_losses[0]:.4f}, last_loss={dpo_losses[-1]:.4f}')


## 10. Inference after DPO and final verification

This is a composition test as much as a quality demo: confirm no errors occurred through pretraining with GQA + RoPE/NoPE + MoE + sliding window + activation checkpointing + ZeRO, then through native SFT and DPO checkpoint hand-offs. Expect limited quality from 52M parameters and any abbreviated run.

In [ ]:
dpo_model.eval()
dpo_outputs = [generate_native(dpo_model, prompt) for prompt in PROMPTS]
for prompt, pretrain, sft, dpo in zip(PROMPTS, pretrain_outputs, sft_outputs, dpo_outputs):
    print('=' * 100)
    print(f'PROMPT: {prompt}')
    print(f'\nPRETRAINED:\n{pretrain}')
    print(f'\nSFT:\n{sft}')
    print(f'\nDPO:\n{dpo}')

print('\nKaggle checklist:')
print('- Hardware reported exactly two T4s and fp16.')
print('- Check logs/native-52m-pretrain/metrics.csv and run.log for persisted pretraining metrics.')
print('- Checkpoint directory contains safetensors plus rank0/rank1 ZeRO optimizer shards.')
print('- Confirm pretraining, SFT, and DPO losses are finite and their displays update.')
print('- This notebook was not executed locally; validate CUDA fp16/GradScaler, NCCL ZeRO, memory use, dataset access, and full-run duration on Kaggle.')
